In [5]:
import os

os.environ["UNSTRUCTURED_API_KEY"] = "3mNKVZRMLxY8rDitgMTdepyFHPaF5v"

In [4]:
from pathlib import Path
from langchain_unstructured import UnstructuredLoader

DATA_DIR = Path("../data/papers")

pdf_files = [str(pdf) for pdf in DATA_DIR.rglob("*.pdf")]

print(f"Found {len(pdf_files)} PDFs")

Found 25 PDFs


In [8]:
loader = UnstructuredLoader(
    file_path=pdf_files,
    partition_via_api=True,
    strategy="fast",
    chunking_strategy="by_title",
    max_characters=1400,
    new_after_n_chars=1100,
    overlap=150,
)

In [9]:
docs = loader.load()
print(len(docs))

# Keep only meaningful content and drop obviously low-signal chunks
import re

LOW_SIGNAL_PATTERNS = [
    r"^references?$",
    r"^bibliography$",
    r"^acknowledg(e|ements?)?$",
    r"^author(s)?$",
    r"^abstract$",
    r"^keywords?$",
]


def clean_chunk_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(?i)\b(author|authors|references|bibliography|acknowledg(e|ements?))\b.*", "", text)
    text = re.sub(r"\b(figure|table|equation)\s*[0-9A-Za-z.-]+\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_low_signal(chunk_text: str) -> bool:
    if not chunk_text:
        return True
    lower = chunk_text.lower()
    if len(chunk_text.split()) < 20:
        return True
    if any(re.search(pattern, lower) for pattern in LOW_SIGNAL_PATTERNS):
        return True
    return False


filtered_docs = []
seen = set()
for doc in docs:
    cleaned = clean_chunk_text(doc.page_content)
    if is_low_signal(cleaned):
        continue
    key = cleaned.lower()
    if len(key) < 80:
        continue
    if key in seen:
        continue
    seen.add(key)
    filtered_docs.append(type(doc)(page_content=cleaned, metadata=doc.metadata))

print(f"Filtered down to {len(filtered_docs)} meaningful chunks")

docs = filtered_docs

INFO: split_pdf event=plan_created operation_id=595a4cd3-ee92-48a4-9b89-fdaf4af722a8 filename=roberta.pdf strategy=fast page_range=1-13 page_count=13 split_size=3 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=595a4cd3-ee92-48a4-9b89-fdaf4af722a8 chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST htt

1883
Filtered down to 1695 meaningful chunks


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

INFO: No device provided, using cpu
INFO: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 

In [11]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [15]:
from hashlib import sha1


def make_chunk_id(text: str) -> str:
    return sha1(text.encode("utf-8")).hexdigest()


chunk_ids = [make_chunk_id(doc.page_content) for doc in docs]
document_ids = vector_store.add_documents(docs, ids=chunk_ids)

print(f"Stored {len(document_ids)} unique chunks.")

# Example retrieval using a more informative query
results = vector_store.similarity_search(
    "How does retrieval augmented generation improve language model performance?",
    k=5,
)
for i, result in enumerate(results, 1):
    print(f"\n[{i}]")
    print(result.page_content)
    print("-" * 100)

Stored 1695 unique chunks.

[1] [12] Aaron Jaech, Adam Kalai, Adam Lerer, Adam Richardson, Ahmed El-Kishky, Aiden Low, Alec Helyar, Aleksander Madry, Alex Beutel, Alex Carney, et al. Openai o1 system card. ArXiv preprint, abs/2412.16720, 2024. URL https://arxiv.org/abs/2412.16720. [13] Zhengbao Jiang, Frank Xu, Luyu Gao, Zhiqing Sun, Qian Liu, Jane Dwivedi-Yu, Yiming Yang, Jamie Callan, and Graham Neubig. Active retrieval augmen...

[2] [29] Pranav Rajpurkar, Jian Zhang, Konstantin Lopyrev, and Percy Liang. SQuAD: 100,000+ questions for machine comprehension of text. In Jian Su, Kevin Duh, and Xavier Carreras, editors, Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing, pages 2383–2392, Austin, Texas, 2016. Association for Computational Linguistics. doi: 10.18653/v1/D16-1264. URL https://aclant...

[3] Large pre-trained language models have been shown to store factual knowledge in their parameters, and achieve state-of-the-art results when ﬁne-tuned 

In [21]:
# Example retrieval using a more informative query
results = vector_store.similarity_search(
    "What is retrieval augmented generation (RAG) ",
    k=4,
)
for i, result in enumerate(results, 1):
    print(f"\n[{i}]")
    print(result)
    print("-" * 100)


[1]
page_content='2.1 RAG Approaches and Systems RAG generally refers to any system where a user query is used to retrieve relevant information from external data sources, whereupon this information is incorporated into the generation of a response to the query by an LLM (or other generative AI model, such as a multi-media model). The query and retrieved records populate a prompt template, which is then passed to the LLM (Ram et al., 2023). RAG is ideal when the total number of records in a data source is too large to include in a single prompt to the LLM, i.e. the amount of text in the data source exceeds the LLM’s context window. In canonical RAG approaches, the retrieval process returns a set number of records that are seman- tically similar to the query and the generated answer uses only the information in those retrieved records. A common approach to conventional RAG is to use text embeddings, retrieving records closest to the query in vector space where closeness corresponds to 

In [22]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert assistant answering questions about NLP research papers.

Use ONLY the context below.

If the answer cannot be found in the context, reply:

"I don't know."

Context:
{context}

Question:
{input}
""")

In [24]:
# Create retriever from vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Create the RAG chain using LCEL
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: retriever.invoke(x["input"])
    )
    | prompt
    | llm
)

In [25]:
response = rag_chain.invoke({"input": "What is Retrieval Augmented Generation"})

# The response is an AIMessage, so access the content directly
print(response.content)

INFO: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Retrieval-Augmented Generation (RAG) is a paradigm that extends the conventional approach of a single retrieval step followed by generation. It involves multi-step iterative retrieval and generation, where the model actively determines when and what to retrieve during the generation process. This approach is designed to improve the performance of language models in knowledge-intensive tasks by leveraging the strengths of both retrieval and generation.


In [26]:
import json
import random
from tqdm import tqdm

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Load benchmark
with open("../data/questions/benchmark.json") as f:
    benchmark = json.load(f)

print(f"Total questions in benchmark: {len(benchmark)}")

# Shuffle with seed for reproducibility
shuffled_benchmark = benchmark.copy()
random.shuffle(shuffled_benchmark)

# Select 50 questions
sample_size = 50
sample_benchmark = shuffled_benchmark[:sample_size]

print(f"Evaluating on {len(sample_benchmark)} shuffled questions (seed={RANDOM_SEED})")
print(f"Sample IDs: {[q['id'] for q in sample_benchmark[:3]]}...")  # Show first 3 IDs

Total questions in benchmark: 500
Evaluating on 50 shuffled questions (seed=42)
Sample IDs: ['generative_adversarial_transformers_q005', 'huggingface_transformers_q004', 'bert_review_q006']...


In [ ]:
import time

# Run RAG pipeline on sampled questions
print("Running RAG pipeline on 50 questions...")
evaluation_data = []
failed_questions = []

for i, item in enumerate(tqdm(sample_benchmark)):
    try:
        # Get retrieved documents
        retrieved_docs = retriever.invoke(item["question"])
        contexts = [doc.page_content for doc in retrieved_docs]
        
        # Get RAG response
        response = rag_chain.invoke({"input": item["question"]})
        response_text = response.content if hasattr(response, 'content') else str(response)
        
        # Collect evaluation data
        evaluation_data.append({
            "user_input": item["question"],
            "retrieved_contexts": contexts,
            "response": response_text,
            "reference": item["ground_truth"],
            "id": item["id"],
        })
        
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        failed_questions.append(item["id"])
        continue
    
    # Rate limiting to avoid API throttling
    if (i + 1) % 10 == 0:
        time.sleep(0.5)

print(f"\nProcessed {len(evaluation_data)} questions successfully")
if failed_questions:
    print(f"Failed on {len(failed_questions)} questions: {failed_questions[:5]}...")

Running RAG pipeline on 50 questions...


  2%|▏         | 1/50 [00:00<00:08,  5.66it/s]

Error on generative_adversarial_transformers_q005: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6174, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


INFO: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
  4%|▍         | 2/50 [00:00<00:08,  5.77it/s]

Error on huggingface_transformers_q004: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6729, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


...

INFO: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
 62%|██████▏   | 31/50 [02:36<00:51,  2.71s/it]

Error on gptq_q012: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6754, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error on dense_text_retrieval_q010: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 9516, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


INFO: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 413 Payload Too Large"
 66%|██████▌   | 33/50 [02:36<00:23,  1.41s/it]

Error on generative_adversarial_transformers_q003: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7101, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error on graph_based_reranking_q004: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 9349, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


INFO: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO: Retrying request to /openai/v1/chat/completions in 55.000000 seconds


In [ ]:
# Workaround: Mock the vertexai import to prevent import errors
import sys
from unittest.mock import MagicMock

sys.modules['google.cloud'] = MagicMock()
sys.modules['google.cloud.aiplatform'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()

# Create Ragas dataset
from ragas import EvaluationDataset

print("Creating Ragas evaluation dataset...")
evaluation_dataset = EvaluationDataset.from_list(evaluation_data)

print(f"Dataset created with {len(evaluation_dataset)} samples")

In [ ]:
# Workaround: Mock the vertexai import to prevent import errors
import sys
from unittest.mock import MagicMock

sys.modules['google.cloud'] = MagicMock()
sys.modules['google.cloud.aiplatform'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()

# Run Ragas evaluation
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.run_config import RunConfig

print("Initializing Ragas evaluator with Groq LLM...")
evaluator_llm = LangchainLLMWrapper(llm)

print("Running Ragas evaluation on 3 metrics...")
print("- LLMContextRecall")
print("- Faithfulness")
print("- FactualCorrectness")
print("\nThis may take several minutes...\n")

# Configure RunConfig to reduce parallelization and increase timeout
run_config = RunConfig(
    max_workers=1,  # Sequential evaluation (no parallelization)
    timeout=120,    # 120 seconds per evaluation
)

results = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)
print(results)

In [ ]:
import pandas as pd

# Convert results to DataFrame for analysis
results_df = results.to_pandas()

print("\nDetailed Metrics Summary:")
print("-" * 80)
print(f"Total samples evaluated: {len(results_df)}")
print(f"\nMetric Scores:")
print(f"  Context Recall average:     {results_df['context_recall'].mean():.4f}")
print(f"  Faithfulness average:       {results_df['faithfulness'].mean():.4f}")
print(f"  Factual Correctness average: {results_df['factual_correctness(mode=f1)'].mean():.4f}")

print(f"\nMetric Ranges:")
print(f"  Context Recall:     [{results_df['context_recall'].min():.4f}, {results_df['context_recall'].max():.4f}]")
print(f"  Faithfulness:       [{results_df['faithfulness'].min():.4f}, {results_df['faithfulness'].max():.4f}]")
print(f"  Factual Correctness: [{results_df['factual_correctness(mode=f1)'].min():.4f}, {results_df['factual_correctness(mode=f1)'].max():.4f}]")

print(f"\nSample Results (first 5):")
print(results_df[['user_input', 'context_recall', 'faithfulness', 'factual_correctness(mode=f1)']].head())

# Save results to CSV for reference
output_path = "../results/evaluation_results_50samples.csv"
results_df.to_csv(output_path, index=False)
print(f"\nResults saved to {output_path}")